In [1]:
import os

import torch
import torch.nn as nn
import yaml
from torch.nn.utils import prune
from torch.quantization import quantize_dynamic
from ultralytics import YOLO

In [2]:
handle_model_yaml = "yolo8_baseline.yaml"
yaml_path = os.path.join(
    os.getenv("HOME_DIR"),
    "config",
    "models",
    handle_model_yaml,
)
with open(yaml_path, "r") as file:
    args = yaml.safe_load(file)

In [3]:
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))
PROCESSED_DIR = os.path.join(os.getenv("HOME_DIR"), "data", "processed")
PROJECT_DIR = os.path.join(
    os.getenv("HOME_DIR"), "results", "models", args["project_results_name"]
)  # Directory for saving results (logs, images, models)
OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "optimized",
)

# **Load model**

In [4]:
model_path = os.path.join(
    PROJECT_DIR,
    "train",
    "weights",
    "best.pt",
)
model = YOLO(model_path, task="detect", verbose=True)

# **Pruning**

In [ ]:
for name, module in model.model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name="weight", amount=0.05)
        prune.remove(module, "weight")

# **Quantization**

In [6]:
model = quantize_dynamic(
    model.model, {torch.nn.Conv2d, torch.nn.Linear}, dtype=torch.qint8
)

/tmp/ipykernel_232077/207176410.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model = quantize_dynamic(


# **Saving model**

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
ckpt = {
    "model": model.model,
    "train_args": {},
}
torch.save(
    ckpt,
    os.path.join(
        OUTPUT_DIR,
        "best_optimized.pt",
    ),
)

In [8]:
yolo_model = YOLO(model_path, task="detect", verbose=True)  # Use original YAML config
yolo_model.model.load_state_dict(model.state_dict())  # Load quantized weights

# Save quantized model with metadata
os.makedirs(OUTPUT_DIR, exist_ok=True)
quantized_path = os.path.join(OUTPUT_DIR, "best_optimized.pt")
yolo_model.save(quantized_path)  # Save with YOLO metadata

# Reload model for export
model = YOLO(quantized_path, task="detect", verbose=True)

# **Export to ONNX**

In [9]:
model.export(
    format="onnx",
    imgsz=IMG_SIZE[0],
    dynamic=False,
)

Ultralytics 8.3.182 🚀 Python-3.13.0 torch-2.8.0+cu128 CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/optimized/best_optimized.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 12, 4725) (6.0 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success ✅ 1.0s, saved as '/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/optimized/best_optimized.onnx' (11.6 MB)

Export complete (1.2s)
Results saved to /home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/optimized
Predict:         yolo predict task=detect model=/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yo

'/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/optimized/best_optimized.onnx'